# LaGCORL pretrained-agent evaluation

Choose `BENCHMARK` once. The notebook builds a fresh randomized evaluation corpus, discovers compatible checkpoints under `logs/`, and evaluates whichever agents are available for that benchmark.

In [ ]:
# Experiment controls — choose the benchmark here, once.
BENCHMARK = "maxcut"  # "maxcut" or "vmp"

## 0. Imports and project paths

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, random, importlib, inspect
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from IPython.display import display

CWD = Path.cwd().resolve()
if (CWD / "LaGCORL" / "gae" / "train_gae.py").exists():
    PROJECT_ROOT = CWD / "LaGCORL"
elif (CWD / "gae" / "train_gae.py").exists():
    PROJECT_ROOT = CWD
else:
    raise FileNotFoundError(f"Cannot find gae/train_gae.py from {CWD}.")

for entry in (PROJECT_ROOT.parent, PROJECT_ROOT, CWD):
    entry = str(entry.resolve())
    if entry not in sys.path:
        sys.path.insert(0, entry)

NOTEBOOK_DIR = CWD
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

# Reload the helper module so notebook edits are reflected before functions are imported.
importlib.invalidate_caches()
import notebook_helpers_api as helpers_api
helpers_api = importlib.reload(helpers_api)

if "vmp_range" not in inspect.signature(helpers_api.sample_benchmark_corpus).parameters:
    raise TypeError(
        "The loaded notebook_helpers_api.sample_benchmark_corpus does not accept "
        "'vmp_range'. Check the printed module path and update/reload that file."
    )

from notebook_helpers_api import (
    sample_benchmark_corpus,
    newest_benchmark_artifact,
    newest_gae_bundle,
    make_pretrained_evaluation_wrapper_factories,
    patch_loaded_wrapper_encoder_references,
    load_pretrained_agents,
    evaluate_pretrained_agents,
    cleanup_temporary_wrapped_corpus,
    summarize_results,
)

SEED = 7
EVAL_EPISODES = 10
EVAL_SEEDS = [101, 102, 103, 104, 105, 106, 107, 108, 109, 110]
LOG_ROOT = PROJECT_ROOT.parent / "logs"
LOG_ROOT.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)


## 1. Select the benchmark and generate a randomized evaluation suite

In [ ]:
# Select once; the evaluation code below is benchmark-independent.
TEST_SCENARIO_COUNT = 5

# MaxCut Params
TEST_NUM_NODES = 20  
TEST_EDGE_WEIGHT_RANGE = (1, 20)

# VMP Params
TEST_VMP_RANGE = {
    "min_VMs": 4,
    "max_VMs": 10,
    "min_PMs": 7,
    "max_PMs": 30,
}

TEST_CORPUS_SEED = 2026
TEST_MAX_SWEEPS = 100

test_corpus = sample_benchmark_corpus(
    PROJECT_ROOT,
    BENCHMARK,
    count=TEST_SCENARIO_COUNT,
    seed=TEST_CORPUS_SEED,
    max_sweeps=TEST_MAX_SWEEPS,
    graph_size_range=(TEST_NUM_NODES, TEST_NUM_NODES),
    edge_weight_range=TEST_EDGE_WEIGHT_RANGE,
    vmp_range=TEST_VMP_RANGE,
)
TEST_CORPUS_DIR = Path(test_corpus["corpus_dir"])
TEST_ENV_IDS = [int(value) for value in test_corpus["graphs"]["id"].tolist()]

print("Benchmark:", BENCHMARK)
print("Test corpus:", TEST_CORPUS_DIR)
print("Scenario count:", len(TEST_ENV_IDS))
display(test_corpus["graphs"].head(10))


## 2. Discover pretrained artifacts

In [ ]:
ARTIFACTS = {
    "discrete": newest_benchmark_artifact(LOG_ROOT, "discrete_ppo", BENCHMARK),
    "gnn_discrete": newest_benchmark_artifact(LOG_ROOT, "discrete_ppo_gnn_padded", BENCHMARK),
    "iterative": newest_benchmark_artifact(LOG_ROOT, "iterative_dqn", BENCHMARK),
    "projection": newest_benchmark_artifact(LOG_ROOT, "projection_ppo", BENCHMARK),
}
GAE_RUN_DIR = newest_gae_bundle(LOG_ROOT, BENCHMARK)

for name, path in ARTIFACTS.items():
    print(f"{name:>12}: {path if path is not None else 'not found'}")
print("         GAE:", GAE_RUN_DIR if GAE_RUN_DIR is not None else "not found")

if not any(path is not None for path in ARTIFACTS.values()):
    raise FileNotFoundError(
        f"No pretrained policies were found for benchmark={BENCHMARK!r} under {LOG_ROOT}. "
        "Run the setup notebook or train the selected benchmark first."
    )


## 3. Configure GAE-backed wrappers when the encoder bundle is available

Wrapper construction and the module-identity compatibility patch are delegated to reusable helpers.

In [ ]:
wrapper_factories = {}
if GAE_RUN_DIR is not None:
    wrapper_factories = make_pretrained_evaluation_wrapper_factories(
        environment=BENCHMARK,
        gae_run_dir=GAE_RUN_DIR,
        project_root=PROJECT_ROOT,
        log_root=LOG_ROOT,
    )
else:
    print(
        "No GAE bundle found. GAE-backed agents will be skipped; "
        "ordinary discrete PPO can still be evaluated."
    )

## 4. Load policies with the correct environment wrappers

In [ ]:
# Ensure every loaded wrapper uses the canonical GAE/VGAE classes.
if wrapper_factories:
    patched_modules = patch_loaded_wrapper_encoder_references()
else:
    patched_modules = []
    print("No GAE-backed wrappers to patch.")


In [ ]:
loaded = load_pretrained_agents(
    artifacts=ARTIFACTS,
    wrapper_factories=wrapper_factories,
    corpus_dir=TEST_CORPUS_DIR,
    env_ids=TEST_ENV_IDS,
    project_root=PROJECT_ROOT,
    log_root=LOG_ROOT,
    environment=BENCHMARK,
)
models = loaded["models"]
load_errors = loaded["load_errors"]
gnn_test_setup = loaded["gnn_test_setup"]


## 5. Evaluate every compatible policy on the same scenarios

The helper evaluates every available policy with the same scenarios and seed list.

In [ ]:
evaluation_root = (
    LOG_ROOT / "pretrained_evaluation"
    / f"{BENCHMARK}_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}"
)
evaluation_results = evaluate_pretrained_agents(
    models=models,
    gnn_test_setup=gnn_test_setup,
    wrapper_factories=wrapper_factories,
    corpus_dir=TEST_CORPUS_DIR,
    env_ids=TEST_ENV_IDS,
    evaluation_root=evaluation_root,
    n_episodes=EVAL_EPISODES,
    seeds=EVAL_SEEDS,
    verbose=0,
)

## 6. Combine, normalize, and export the results

In [ ]:
comparison_results = summarize_results(
    evaluation_results,
    benchmark=BENCHMARK,
    evaluation_root=evaluation_root,
)

comparison_table = comparison_results["comparison_table"]
summary_table = comparison_results["summary_table"]
all_episodes = comparison_results["all_episodes"]